# Notebook 32 — Final IMERG precipitation-convergence analysis

This is Phase 3. It makes no ERA5 or NASA network requests. It reads the compact event data backed up by Notebook 31, verifies that every event available in IMERG Final V07 is complete, then creates the four scatterplots and the correlation/regression table. Events outside the current Final-V07 archive are retained in the inventory as documented exclusions and are not mixed with a different IMERG product.

The predictor is the saved convergence for the digitized Shinoda JPCZ polygon. The two precipitation regions are the same Shinoda polygon and the additional coastal wedge.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
# Accept a raw Git URL even if it was accidentally pasted as a Markdown link.
if REPO_URL.startswith('[') and '](' in REPO_URL and REPO_URL.endswith(')'):
    REPO_URL = REPO_URL.rsplit('](', 1)[1][:-1]
if not REPO_URL.startswith('https://'):
    raise ValueError(f'REPO_URL must be a raw https Git URL, not {REPO_URL!r}')

from google.colab import drive
drive.mount('/content/drive')
if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    clone = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError(f'Git clone failed for {REPO_URL}:\n{clone.stderr}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from jpcz_catalog.imerg_workflow import association_statistics, atomic_csv, read_checkpoint, write_event_plan

ALLOW_PARTIAL_ANALYSIS = False
CONFIDENCE_LEVEL = 0.95  # Pre-specified two-sided confidence level; alpha = 0.05.
DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
PLAN_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_collection_plan.csv'
EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
ANALYSIS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_precipitation_convergence_metrics.csv'
LOG_ANALYSIS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_log1p_event_precipitation_convergence_metrics.csv'
STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics.csv'
LOG_STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_log1p_accumulation_association_statistics.csv'
FIELD_GUIDE_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics_field_guide.csv'
PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_scatter.png'
LOG_PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_log1p_accumulation_scatter.png'
R_CI_PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_accumulation_correlation_95ci.png'
QUARTILE_SUMMARY_PATH = DRIVE_ANALYSIS_DIR / 'imerg_accumulation_convergence_quartile_summary.csv'
QUARTILE_PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_accumulation_convergence_quartile_means.png'
LOG_ACCUMULATION_OFFSET_MM = 1.0

if not PLAN_PATH.exists():
    raise FileNotFoundError('The collection plan is missing. Run Notebook 30, then collect data with Notebook 31.')
plan = pd.read_csv(PLAN_PATH, parse_dates=['event_start', 'event_end', 'event_peak', 'precip_window_start', 'precip_window_end_exclusive'])
event_metrics = read_checkpoint(EVENT_PATH, parse_dates=('event_peak',))
if not {'event_id', 'event_peak'}.issubset(event_metrics.columns):
    event_metrics = pd.DataFrame(columns=['event_id', 'event_peak'])
# Apply the current Final-V07 availability policy even if this Drive plan was created by an older notebook version.
plan = write_event_plan(plan, event_metrics, path=PLAN_PATH)
inventory = plan.merge(event_metrics, on=['event_id', 'event_peak'], how='left')
atomic_csv(inventory, ANALYSIS_PATH)
available_plan = plan.loc[plan['analysis_inclusion'].eq('include')].copy()
analysis = inventory.loc[inventory['analysis_inclusion'].eq('include')].copy()
# log1p preserves zero-accumulation events: log(1 + accumulation / 1 mm).
for region in ('jpcz_polygon', 'coastal_wedge'):
    accumulation_column = f'{region}_imerg_accumulation_mm'
    if (analysis[accumulation_column].dropna() < 0).any():
        raise ValueError(f'Negative IMERG accumulation found in {accumulation_column}.')
    analysis[f'{region}_imerg_log1p_accumulation'] = np.log1p(analysis[accumulation_column] / LOG_ACCUMULATION_OFFSET_MM)
atomic_csv(analysis, LOG_ANALYSIS_PATH)
complete_events = int((available_plan['collection_status'] == 'complete').sum())
unavailable_events = int((plan['analysis_inclusion'] == 'exclude').sum())
all_data_ready = complete_events == len(available_plan) and len(available_plan) > 0
print(f'Final V07 analysis inventory: {complete_events}/{len(available_plan)} available events complete; {unavailable_events} catalog events excluded because Final V07 is unavailable for their windows.')
print(f'Pre-specified two-sided confidence level: {CONFIDENCE_LEVEL:.0%} (alpha = {1 - CONFIDENCE_LEVEL:.2f}).')
print('Final-analysis readiness:', 'READY' if all_data_ready else 'NOT READY — resume Notebook 31.')
display(plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(analysis.head())

In [ ]:
specifications = [
    ('Shinoda JPCZ polygon', 'IMERG accumulation (mm)', 'jpcz_polygon_convergence_1e5_s-1', 'jpcz_polygon_imerg_accumulation_mm'),
    ('Coastal wedge', 'IMERG accumulation (mm)', 'jpcz_polygon_convergence_1e5_s-1', 'coastal_wedge_imerg_accumulation_mm'),
]
log_specifications = [
    ('Shinoda JPCZ polygon', 'ln[1 + IMERG accumulation (mm) / 1 mm]', 'jpcz_polygon_convergence_1e5_s-1', 'jpcz_polygon_imerg_log1p_accumulation'),
    ('Coastal wedge', 'ln[1 + IMERG accumulation (mm) / 1 mm]', 'jpcz_polygon_convergence_1e5_s-1', 'coastal_wedge_imerg_log1p_accumulation'),
]

if not all_data_ready and not ALLOW_PARTIAL_ANALYSIS:
    statistics_table = pd.DataFrame([
        {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'waiting for complete Drive inventory'}
        for region, measure, _, _ in specifications
    ])
    print('Final statistics remain locked until all Final-V07-available events are saved.')
else:
    statistics_table = pd.DataFrame([
        association_statistics(analysis, x_column=x, y_column=y, region=region, measure=measure, confidence_level=CONFIDENCE_LEVEL)
        for region, measure, x, y in specifications
    ])
atomic_csv(statistics_table, STATS_PATH)
presentation_columns = [
    'region', 'precipitation_measure', 'n', 'confidence_level', 'alpha',
    'x_mean', 'x_sample_sd', 'y_mean', 'y_sample_sd',
    'pearson_r', 'r_95ci_low', 'r_95ci_high', 'r_two_sided_p',
    'slope', 'slope_standard_error', 'slope_95ci_low', 'slope_95ci_high',
    'intercept', 'intercept_standard_error', 'r_squared',
    'rmse', 'residual_standard_error', 'regression_df',
    'evidence_for_nonzero_association_alpha_0.05', 'status',
]
field_guide = pd.DataFrame([
    ('x_mean / x_sample_sd', 'Mean and n−1 sample SD of convergence strength C = −D12 (10^-5 s^-1).'),
    ('y_mean / y_sample_sd', 'Mean and n−1 sample SD of the stated precipitation response; its units follow the response label.'),
    ('pearson_r', 'One signed Pearson correlation coefficient. Positive means stronger C = −D12 is associated with greater precipitation.'),
    ('r_95ci_low / high', 'The 95% CI error-bar endpoints for pearson_r, calculated with Fisher z; SE_z = 1/sqrt(n−3). An interval crossing zero is not significant at two-sided α = 0.05.'),
    ('slope_95ci_low / high', '95% OLS slope interval: slope ± t_(0.975, n−2) × slope standard error.'),
    ('shaded regression band', 'Two-sided 95% confidence interval for the fitted mean precipitation response at each convergence value; it is not a prediction interval for individual events.'),
    ('rmse', 'Root mean squared vertical residual; in the same units as the precipitation response.'),
    ('residual_standard_error', 'Residual SD estimated with n−2 regression degrees of freedom.'),
    ('log1p accumulation', 'Exploratory transform ln[1 + accumulation/(1 mm)]. It retains zero-accumulation events and reduces the influence of large totals.'),
    ('convergence quartiles', 'Four equal-count groups defined from C = −D12 only. Vertical bars show the two-sided 95% CI of each group mean accumulation, not SD of individual events.'),
], columns=['field', 'definition'])
atomic_csv(field_guide, FIELD_GUIDE_PATH)
print('Detailed statistics saved:', STATS_PATH)
print('Field definitions saved:', FIELD_GUIDE_PATH)
display(statistics_table.reindex(columns=presentation_columns).round(4))
display(field_guide)

if not all_data_ready and not ALLOW_PARTIAL_ANALYSIS:
    log_statistics_table = pd.DataFrame([
        {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'waiting for complete Drive inventory'}
        for region, measure, _, _ in log_specifications
    ])
else:
    log_statistics_table = pd.DataFrame([
        association_statistics(analysis, x_column=x, y_column=y, region=region, measure=measure, confidence_level=CONFIDENCE_LEVEL)
        for region, measure, x, y in log_specifications
    ])
atomic_csv(log_statistics_table, LOG_STATS_PATH)
print('Log-transformed accumulation statistics saved:', LOG_STATS_PATH)
display(log_statistics_table.reindex(columns=presentation_columns).round(4))

In [ ]:
def plot_association(ax, frame, x_column, y_column, title, ylabel, summary):
    if summary.get('status') != 'ok':
        ax.text(0.5, 0.5, 'Waiting for complete IMERG event inventory', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return None
    sample = frame[[x_column, y_column, 'duration_hours']].dropna()
    points = ax.scatter(sample[x_column], sample[y_column], c=sample['duration_hours'], cmap='viridis', s=40, alpha=0.85, edgecolor='white', linewidth=0.35)
    fit = stats.linregress(sample[x_column], sample[y_column])
    xline = np.linspace(sample[x_column].min(), sample[x_column].max(), 100)
    yline = fit.intercept + fit.slope * xline
    residuals = sample[y_column] - (fit.intercept + fit.slope * sample[x_column])
    residual_standard_error = np.sqrt(np.sum(residuals**2) / (len(sample) - 2))
    x_mean = sample[x_column].mean()
    x_centered_sum_squares = np.sum((sample[x_column] - x_mean)**2)
    confidence_level = float(summary.get('confidence_level', CONFIDENCE_LEVEL))
    t_critical = stats.t.ppf(1 - (1 - confidence_level) / 2, len(sample) - 2)
    mean_response_se = residual_standard_error * np.sqrt(1 / len(sample) + (xline - x_mean)**2 / x_centered_sum_squares)
    ax.fill_between(xline, yline - t_critical * mean_response_se, yline + t_critical * mean_response_se, color='#c0392b', alpha=0.14, label=f'{confidence_level:.0%} CI of fitted mean')
    ax.plot(xline, yline, color='#c0392b', linewidth=2, label='OLS fit')
    ax.set_title(title)
    ax.set_xlabel('925-hPa convergence strength, C = −D12 (10^-5 s^-1)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(loc='lower right', fontsize=7, frameon=True)
    statistics_text = (
        f"n={int(summary['n'])}\n"
        f"r={summary['pearson_r']:+.2f} [95% CI {summary['r_95ci_low']:+.2f}, {summary['r_95ci_high']:+.2f}]\n"
        f"slope={summary['slope']:+.2f} [95% CI {summary['slope_95ci_low']:+.2f}, {summary['slope_95ci_high']:+.2f}]\n"
        f"RMSE={summary['rmse']:.2f}; p={summary['r_two_sided_p']:.3g}"
    )
    ax.text(0.03, 0.97, statistics_text, va='top', transform=ax.transAxes, fontsize=8.4, linespacing=1.2, bbox={'facecolor': 'white', 'alpha': 0.94, 'edgecolor': '#555555'})
    return points

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), constrained_layout=True)
last_points = None
for ax, (region, measure, x_column, y_column) in zip(axes.flat, specifications):
    summary = statistics_table.loc[(statistics_table['region'] == region) & (statistics_table['precipitation_measure'] == measure)].iloc[0].to_dict()
    result = plot_association(ax, analysis, x_column, y_column, f'{region}: {measure}', measure, summary)
    if result is not None:
        last_points = result
if last_points is not None:
    fig.colorbar(last_points, ax=axes, shrink=0.82, pad=0.02, label='Merged-event duration (h)')
fig.suptitle('IMERG Final V07 event accumulation versus JPCZ convergence strength (C = −D12)', fontsize=14)
fig.savefig(PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# Exploratory sensitivity: compress the highly skewed accumulation distribution.
fig_log, axes_log = plt.subplots(1, 2, figsize=(14, 5.6), constrained_layout=True)
last_log_points = None
for ax, (region, measure, x_column, y_column) in zip(axes_log.flat, log_specifications):
    summary = log_statistics_table.loc[(log_statistics_table['region'] == region) & (log_statistics_table['precipitation_measure'] == measure)].iloc[0].to_dict()
    result = plot_association(ax, analysis, x_column, y_column, f'{region}: log-transformed IMERG accumulation', measure, summary)
    if result is not None:
        last_log_points = result
if last_log_points is not None:
    fig_log.colorbar(last_log_points, ax=axes_log, shrink=0.82, pad=0.02, label='Merged-event duration (h)')
fig_log.suptitle('Exploratory log-transformed event accumulation versus JPCZ convergence strength (C = −D12)', fontsize=14)
fig_log.savefig(LOG_PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# One signed r point per comparison; horizontal bars are its Fisher-z 95% confidence interval.
r_ci_summary = pd.concat([
    statistics_table.assign(display_label=['Shinoda polygon\naccumulation', 'Coastal wedge\naccumulation']),
    log_statistics_table.assign(display_label=['Shinoda polygon\nln(1 + accumulation)', 'Coastal wedge\nln(1 + accumulation)']),
], ignore_index=True)
r_ci_summary = r_ci_summary.loc[r_ci_summary['status'].eq('ok')].reset_index(drop=True)
if r_ci_summary.empty:
    raise RuntimeError('No completed correlation results are available for the 95% CI summary.')
y_positions = np.arange(len(r_ci_summary))[::-1]
left_error = r_ci_summary['pearson_r'] - r_ci_summary['r_95ci_low']
right_error = r_ci_summary['r_95ci_high'] - r_ci_summary['pearson_r']
fig_r_ci, ax_r_ci = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
ax_r_ci.errorbar(r_ci_summary['pearson_r'], y_positions, xerr=np.vstack([left_error, right_error]), fmt='o', color='#c0392b', ecolor='#2c3e50', capsize=5, markersize=7)
ax_r_ci.axvline(0, color='black', linewidth=1, linestyle='--', label='No linear association (r = 0)')
ax_r_ci.set_yticks(y_positions, r_ci_summary['display_label'])
ax_r_ci.set_xlabel('Pearson correlation coefficient, r (horizontal bars = 95% CI)')
ax_r_ci.set_title('Accumulation–convergence correlations with 95% confidence intervals')
ax_r_ci.grid(axis='x', alpha=0.25)
ax_r_ci.legend(loc='lower right')
fig_r_ci.savefig(R_CI_PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# Descriptive convergence-bin comparison: quartiles are defined from C = −D12 only, never from precipitation.
BIN_LABELS = ['Weakest convergence', 'Weak–moderate convergence', 'Moderate–strong convergence', 'Strongest convergence']
BIN_SHORT_LABELS = ['Weakest', 'Weak–moderate', 'Moderate–strong', 'Strongest']
CONVERGENCE_COLUMN = 'jpcz_polygon_convergence_1e5_s-1'
quartile_frame = analysis[[CONVERGENCE_COLUMN, 'jpcz_polygon_imerg_accumulation_mm', 'coastal_wedge_imerg_accumulation_mm']].copy()
quartile_frame['convergence_group'] = pd.qcut(quartile_frame[CONVERGENCE_COLUMN], q=4, labels=BIN_LABELS)
quartile_rows = []
for group_label, group in quartile_frame.groupby('convergence_group', observed=False):
    c_values = group[CONVERGENCE_COLUMN].dropna()
    for region, response_column in [('Shinoda JPCZ polygon', 'jpcz_polygon_imerg_accumulation_mm'), ('Coastal wedge', 'coastal_wedge_imerg_accumulation_mm')]:
        values = group[response_column].dropna()
        n_group = len(values)
        mean = values.mean()
        sample_sd = values.std(ddof=1)
        sem = sample_sd / np.sqrt(n_group)
        t_critical = stats.t.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2, n_group - 1)
        ci_half_width = t_critical * sem
        quartile_rows.append({
            'region': region, 'convergence_group': str(group_label), 'n_events': n_group,
            'C_group_min_1e5_s-1': c_values.min(), 'C_group_max_1e5_s-1': c_values.max(), 'C_group_mean_1e5_s-1': c_values.mean(),
            'mean_accumulation_mm': mean, 'sample_sd_mm': sample_sd, 'sem_mm': sem,
            'confidence_level': CONFIDENCE_LEVEL, 'mean_95ci_low_mm': mean - ci_half_width, 'mean_95ci_high_mm': mean + ci_half_width,
        })
quartile_summary = pd.DataFrame(quartile_rows)
quartile_summary['convergence_group'] = pd.Categorical(quartile_summary['convergence_group'], categories=BIN_LABELS, ordered=True)
quartile_summary = quartile_summary.sort_values(['region', 'convergence_group']).reset_index(drop=True)
atomic_csv(quartile_summary, QUARTILE_SUMMARY_PATH)
print('Convergence-quartile summary saved:', QUARTILE_SUMMARY_PATH)
display(quartile_summary.round(4))

fig_bins, axes_bins = plt.subplots(1, 2, figsize=(14, 6.3), sharey=True, constrained_layout=True)
for ax, region in zip(axes_bins.flat, ['Shinoda JPCZ polygon', 'Coastal wedge']):
    plot_data = quartile_summary.loc[quartile_summary['region'].eq(region)].copy().sort_values('convergence_group')
    positions = np.arange(len(plot_data))
    lower_error = plot_data['mean_accumulation_mm'] - plot_data['mean_95ci_low_mm']
    upper_error = plot_data['mean_95ci_high_mm'] - plot_data['mean_accumulation_mm']
    range_tick_labels = [f'{short}\nC={low:.2f}–{high:.2f}' for short, low, high in zip(BIN_SHORT_LABELS, plot_data['C_group_min_1e5_s-1'], plot_data['C_group_max_1e5_s-1'])]
    ax.bar(positions, plot_data['mean_accumulation_mm'], color='#4c78a8', alpha=0.82, width=0.68)
    ax.errorbar(positions, plot_data['mean_accumulation_mm'], yerr=np.vstack([lower_error, upper_error]), fmt='none', ecolor='black', capsize=5, linewidth=1.4, label='95% CI of group mean')
    ax.set_xticks(positions, range_tick_labels, fontsize=9)
    ax.set_title(region)
    ax.set_xlabel('Observed C = −D12 range within convergence quartile (10^-5 s^-1)')
    ax.set_ylabel('Mean IMERG event accumulation (mm)')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)
fig_bins.suptitle('Mean event accumulation by convergence-strength quartile\nTick labels give the observed C = −D12 range in 10^-5 s^-1; bars = group means; vertical error bars = two-sided 95% CI.', fontsize=13)
fig_bins.savefig(QUARTILE_PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()

## Methods wording

For each merged JPCZ episode, the catalog retains the detector's saved 12-hour trailing, area-weighted 925-hPa divergence at the event peak, denoted D12. In the detector and catalog, D12 < 0 is convergence and D12 > 0 is divergence. For the association plots and regression only, we define convergence strength as C = −D12, so larger plotted values mean stronger convergence; the detector and its thresholds remain based on unmodified divergence. We obtained GPM IMERG Final V07 gauge-calibrated precipitation (`precipitation`; half-hourly 0.1 degree grid, with a legacy `precipitationCal` fallback only if present), calculated cosine-latitude-area-weighted precipitation rates over the Shinoda polygon and coastal wedge, and calculated event accumulation by summing rate times 0.5 hour over each merged-event window. The primary analysis uses untransformed accumulation in mm. As an exploratory distributional sensitivity analysis, we additionally use ln[1 + accumulation/(1 mm)], which retains zero-accumulation events while reducing the leverage of very large totals; it is not a replacement for the pre-specified primary analysis. For each association, we report the mean and n−1 sample SD of both variables, one signed two-sided Pearson r, a 95% r confidence interval based on the Fisher-z approximation (SE_z = 1/sqrt(n−3)), and ordinary least-squares slope, slope SE, slope 95% interval based on t_(0.975, n−2), R², residual standard error, RMSE, and two-sided p. Each scatterplot additionally shows the two-sided 95% confidence band for the fitted mean precipitation response; it is not an interval for individual events. The correlation-coefficient summary plots each signed r as one point with its 95% CI as a horizontal error bar; an interval crossing r = 0 is not significant at two-sided α = 0.05. As a descriptive complement, events are also split into four sample-relative equal-count convergence quartiles using C = −D12 only: weakest, weak–moderate, moderate–strong, and strongest convergence. For each quartile and region, we plot mean accumulation with a vertical two-sided 95% CI for that group mean. This grouped display is not an additional correlation-significance test. The test evaluates whether the population linear association is distinguishable from zero; it does not establish causation.